<!-- paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 참고 구현입니다. 필요한 한 줄만 확인하고 다시 실습 창으로 돌아가세요.

# 21. GPU 학습·최적화 실습 — RTX 3080에서도 안전하게

이 페어 랩은 GPU를 단순히 "사용"하는 수준을 넘어, 데이터 이동·AMP·gradient accumulation,
clipping, 메모리 관찰, checkpoint/resume, 재현성까지 하나의 작은 학습 시스템으로 묶습니다.
JupyterLab 왼쪽에는 exercise, 오른쪽에는 solution을 열고 TODO를 한 셀씩 완성하세요.

In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 안전 원칙과 완료 기준

기본 실험은 CPU에서도 빠르게 끝나는 synthetic classification입니다. CUDA가 있으면 pinned
memory와 AMP를 자동 사용합니다. Batch-size 탐색은 **분석적 메모리 추정만 수행**해 실제 OOM을
유발하지 않습니다. RTX 3080 16GB에서도 전체 VRAM을 목표로 하지 않고 다른 앱과 CUDA context를
위해 여유를 둡니다.

완료 기준: device 확인 → 재현 가능한 loader → non-blocking 이동 → AMP 학습 → peak memory →
안전 batch 후보 → checkpoint 재개 → 조건부 `torch.compile` → 평가를 모두 통과하는 것입니다.

In [ ]:
import math
import os
import random
import tempfile
import time
from contextlib import nullcontext
from pathlib import Path

import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset
from llm_engineering_lab.acceleration import get_accelerator

SEED = 21
ACCELERATOR = get_accelerator()
device = ACCELERATOR.device
print(f"torch={torch.__version__}, {ACCELERATOR.summary()}")

## 1) Hardware inventory와 메모리 예산

코드에서 device를 문자열로 가정하지 말고 실제 가용성을 확인합니다. CUDA일 때 GPU 이름,
compute capability, 총 VRAM을 기록하세요. 총 VRAM은 곧 사용 가능 VRAM이 아닙니다. Driver,
display, 다른 프로세스, CUDA context가 함께 사용하므로 이 랩은 총량의 65%만 탐색 예산으로 둡니다.

In [ ]:
if device.type == "cuda":
    properties = torch.cuda.get_device_properties(device)
    hardware = {
        "name": properties.name,
        "total_vram_gb": properties.total_memory / 1024**3,
        "compute_capability": f"{properties.major}.{properties.minor}",
        "safe_budget_gb": properties.total_memory * 0.65 / 1024**3,
    }
else:
    hardware = {
        "name": "CPU fallback",
        "total_vram_gb": 0.0,
        "compute_capability": None,
        "safe_budget_gb": 0.5,
    }
print(hardware)

## 2) 재현성 설정

Python과 PyTorch RNG를 함께 고정하고, CUDA라면 모든 GPU seed를 설정합니다. Deterministic
algorithm은 느려지거나 일부 연산에서 경고가 날 수 있어 `warn_only=True`로 학습 실습을 막지
않습니다. 완전한 재현에는 데이터·코드·라이브러리·driver 버전까지 함께 기록해야 합니다.

In [ ]:
def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.benchmark = False


seed_everything(SEED)
print("seeded:", SEED)

## 3) 학습 가능한 synthetic dataset

입력 feature에 고정된 teacher matrix를 곱해 class를 만듭니다. Train/validation이 같은 teacher를
공유하므로 일반화를 측정할 수 있고, 파일 I/O 없이 GPU 파이프라인만 빠르게 반복합니다. Dataset은
CPU tensor를 반환합니다. Worker가 GPU tensor를 만드는 패턴은 CUDA context 충돌의 원인이 됩니다.

In [ ]:
class SyntheticClassificationDataset(Dataset):
    def __init__(
        self,
        size: int,
        input_dim: int,
        teacher: Tensor,
        seed: int,
    ) -> None:
        generator = torch.Generator().manual_seed(seed)
        self.features = torch.randn(size, input_dim, generator=generator)
        noise = 0.15 * torch.randn(size, teacher.shape[1], generator=generator)
        self.labels = (self.features @ teacher + noise).argmax(dim=1)

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor]:
        return self.features[index], self.labels[index]

## 4) Windows/Jupyter에 안전한 DataLoader

CUDA일 때 `pin_memory=True`인 CPU batch를 `non_blocking=True`로 복사하면 compute와 transfer를
겹칠 가능성이 생깁니다. Windows Notebook에서는 `num_workers>0`가 셀 코드를 다시 import하거나
worker 시작 문제를 만들 수 있어 기본은 0입니다. 별도 `.py` 진입점에서만 worker 수를 조정하세요.

In [ ]:
input_dim, hidden_dim, num_classes = 128, 256, 8
teacher_generator = torch.Generator().manual_seed(999)
teacher = torch.randn(input_dim, num_classes, generator=teacher_generator)
train_dataset = SyntheticClassificationDataset(2048, input_dim, teacher, seed=101)
validation_dataset = SyntheticClassificationDataset(512, input_dim, teacher, seed=202)

batch_size = 128
loader_generator = torch.Generator().manual_seed(SEED)
pin_memory = device.type == "cuda"
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=loader_generator,
    num_workers=0,
    pin_memory=pin_memory,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory,
)
print("batches:", len(train_loader), "pin_memory:", pin_memory)

## 5) Host-to-device 경계 한곳에 모으기

Batch 이동을 helper 하나로 모으면 model 코드에 `.cuda()`가 흩어지지 않고 CPU fallback도
유지됩니다. `non_blocking=True`는 source가 pinned memory일 때 비동기 전송 가능성을 열어 주며,
즉시 동기화된다는 보장은 아닙니다.

In [ ]:
def move_batch(batch: tuple[Tensor, Tensor]) -> tuple[Tensor, Tensor]:
    non_blocking = device.type == "cuda"
    return tuple(tensor.to(device, non_blocking=non_blocking) for tensor in batch)


sample_features, sample_labels = move_batch(next(iter(train_loader)))
assert sample_features.device.type == sample_labels.device.type == device.type
print(tuple(sample_features.shape), sample_features.device)

## 6) 작은 모델과 sanity check

먼저 한 batch에서 forward/loss/backward shape를 확인합니다. GPU 최적화 전에 잘못된 label dtype,
output class 수, device mismatch를 잡는 가장 싼 테스트입니다. 이 모델은 의도적으로 작아 노트북을
막지 않으며, 동일한 학습 loop를 CNN/Transformer로 교체할 수 있습니다.

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, features: Tensor) -> Tensor:
        return self.network(features)


model = TinyClassifier(input_dim, hidden_dim, num_classes).to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"parameters={parameter_count:,}")

In [ ]:
sanity_logits = model(sample_features)
sanity_loss = nn.functional.cross_entropy(sanity_logits, sample_labels)
sanity_loss.backward()
assert sanity_logits.shape == (len(sample_labels), num_classes)
assert torch.isfinite(sanity_loss)
model.zero_grad(set_to_none=True)
print("sanity loss:", round(sanity_loss.item(), 4))

## 7) GPU memory 측정

`allocated`는 tensor가 실제 사용 중인 메모리, `reserved`는 allocator가 재사용하려고 확보한
메모리, `peak`는 reset 이후 최대 allocated입니다. `empty_cache()`는 살아 있는 tensor를 지우지
않으며 매 step 호출하면 오히려 느려질 수 있습니다. 측정 경계에서만 synchronize합니다.

In [ ]:
def reset_peak_memory() -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)


def memory_snapshot() -> dict[str, float]:
    if device.type != "cuda":
        return {"allocated_mb": 0.0, "reserved_mb": 0.0, "peak_mb": 0.0}
    torch.cuda.synchronize(device)
    scale = 1024**2
    return {
        "allocated_mb": torch.cuda.memory_allocated(device) / scale,
        "reserved_mb": torch.cuda.memory_reserved(device) / scale,
        "peak_mb": torch.cuda.max_memory_allocated(device) / scale,
    }


print("before training:", memory_snapshot())

## 8) AMP autocast + GradScaler

Autocast 영역에는 forward와 loss만 넣습니다. Backward는 autocast 밖에서 scaler를 통해 실행합니다.
Clipping 전에 반드시 `scaler.unscale_(optimizer)`를 호출해야 실제 gradient norm을 봅니다.
CPU fallback에서는 `nullcontext`와 disabled scaler로 같은 코드를 유지합니다.

In [ ]:
amp_enabled = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)


def amp_context():
    if amp_enabled:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


print("AMP enabled:", amp_enabled)

## 9) Accumulation + clipping + throughput 학습 함수

Effective batch는 `micro batch × accumulation_steps`입니다. Loss를 accumulation 수로 나누고,
optimizer/scheduler/zero_grad는 update 시점에만 호출합니다. 마지막 남은 micro-batch도 update하며
clipping 전 norm과 examples/sec를 함께 기록해 안정성과 성능을 비교합니다.

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler,
    accumulation_steps: int = 2,
    clip_norm: float = 1.0,
) -> dict[str, float]:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    reset_peak_memory()
    start = time.perf_counter()
    total_loss = 0.0
    correct = 0
    examples = 0
    updates = 0
    last_grad_norm = 0.0

    for micro_step, batch in enumerate(loader):
        features, labels = move_batch(batch)
        with amp_context():
            logits = model(features)
            raw_loss = nn.functional.cross_entropy(logits, labels)
            loss = raw_loss / accumulation_steps
        scaler.scale(loss).backward()
        total_loss += raw_loss.detach().item() * len(labels)
        correct += logits.detach().argmax(dim=1).eq(labels).sum().item()
        examples += len(labels)

        update_now = (micro_step + 1) % accumulation_steps == 0
        update_now |= micro_step + 1 == len(loader)
        if update_now:
            scaler.unscale_(optimizer)
            grad_norm = nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            last_grad_norm = float(grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            updates += 1

    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - start
    return {
        "loss": total_loss / examples,
        "accuracy": correct / examples,
        "examples_per_second": examples / max(elapsed, 1e-9),
        "updates": float(updates),
        "last_grad_norm": last_grad_norm,
        **memory_snapshot(),
    }

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.01)
first_epoch_metrics = train_one_epoch(
    model, train_loader, optimizer, scaler, accumulation_steps=2, clip_norm=1.0
)
assert first_epoch_metrics["updates"] == math.ceil(len(train_loader) / 2)
print({key: round(value, 3) for key, value in first_epoch_metrics.items()})

## 10) OOM 없는 batch-size 탐색

후보 batch를 실제로 할당하며 OOM을 잡는 코드는 allocator fragmentation과 Notebook kernel 불안정을
만들 수 있습니다. 먼저 parameter/optimizer/gradient와 activation을 보수적으로 추정해 후보를
거릅니다. 이 값은 **상한 보장치가 아닌 1차 필터**이며, 실제 모델에서는 profiler와 작은 증가
실험으로 확인하세요. Transformer activation은 sequence length 제곱 항도 포함해야 합니다.

In [ ]:
bytes_per_parameter = next(model.parameters()).element_size()
parameter_bytes = sum(p.numel() * p.element_size() for p in model.parameters())


def estimate_training_bytes(candidate_batch: int) -> int:
    model_gradient_optimizer = parameter_bytes * 4  # weight + grad + Adam m/v
    # 여러 layer의 저장 activation과 framework overhead를 보수적으로 확대합니다.
    activation_values = candidate_batch * (input_dim + hidden_dim * 6 + num_classes)
    activation_bytes = activation_values * bytes_per_parameter * 8
    fixed_overhead = 128 * 1024**2
    return model_gradient_optimizer + activation_bytes + fixed_overhead


candidate_batches = [64, 128, 256, 512, 1024, 2048]
if device.type == "cuda":
    budget_bytes = int(torch.cuda.get_device_properties(device).total_memory * 0.65)
else:
    budget_bytes = 512 * 1024**2
safe_candidates = [
    batch
    for batch in candidate_batches
    if estimate_training_bytes(batch) < budget_bytes
]
estimate_table = {
    batch: round(estimate_training_bytes(batch) / 1024**2, 1)
    for batch in candidate_batches
}
assert safe_candidates
print("estimated MB:", estimate_table)
print("analysis-only safe candidates:", safe_candidates, "(실제 할당하지 않음)")

## 11) Checkpoint 저장과 resume

Model만 저장하면 정확한 재개가 아닙니다. Optimizer, GradScaler, epoch/global step, RNG, DataLoader
generator 상태를 함께 저장합니다. 새 객체에 로드한 뒤 실제로 다음 epoch를 진행하여 resume 경로를
검증합니다. 실무에서는 scheduler, tokenizer/config, git commit, metric도 포함하세요.

In [ ]:
checkpoint_path = Path(tempfile.gettempdir()) / "ai_lab21_gpu_resume_solution.pt"
checkpoint_payload = {
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "scaler_state": scaler.state_dict(),
    "epoch": 1,
    "global_updates": int(first_epoch_metrics["updates"]),
    "rng_cpu": torch.get_rng_state(),
    "rng_cuda": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
    "loader_generator_state": loader_generator.get_state(),
    "metrics": first_epoch_metrics,
}
torch.save(checkpoint_payload, checkpoint_path)
assert checkpoint_path.exists() and checkpoint_path.stat().st_size > 0
print("checkpoint bytes:", checkpoint_path.stat().st_size)

In [ ]:
payload = torch.load(checkpoint_path, map_location=device, weights_only=True)
resumed_model = TinyClassifier(input_dim, hidden_dim, num_classes).to(device)
resumed_model.load_state_dict(payload["model_state"])
resumed_optimizer = torch.optim.AdamW(
    resumed_model.parameters(), lr=2e-3, weight_decay=0.01
)
resumed_optimizer.load_state_dict(payload["optimizer_state"])
resumed_scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
resumed_scaler.load_state_dict(payload["scaler_state"])
torch.set_rng_state(payload["rng_cpu"].cpu())
if device.type == "cuda" and payload["rng_cuda"] is not None:
    torch.cuda.set_rng_state_all([state.cpu() for state in payload["rng_cuda"]])
loader_generator.set_state(payload["loader_generator_state"].cpu())

second_epoch_metrics = train_one_epoch(
    resumed_model,
    train_loader,
    resumed_optimizer,
    resumed_scaler,
    accumulation_steps=2,
    clip_norm=1.0,
)
total_updates = payload["global_updates"] + int(second_epoch_metrics["updates"])
assert total_updates == 2 * math.ceil(len(train_loader) / 2)
print(
    "resumed epoch metrics:", {k: round(v, 3) for k, v in second_epoch_metrics.items()}
)

## 12) `torch.compile`은 명시적 opt-in

Compile은 첫 호출 비용, graph break, backend/driver 조합 이슈가 있어 작은 Notebook에서는 오히려
느릴 수 있습니다. 기본은 꺼 두고 CUDA, API 존재, 사용자의 명시적 선택을 모두 만족할 때만
실행합니다. 먼저 eager correctness를 통과하고 고정 shape의 반복 workload에서 warmup 이후를
benchmark하세요. 문제가 나면 원본 model로 즉시 fallback합니다.

In [ ]:
RUN_TORCH_COMPILE = False
inference_model = resumed_model
compile_status = "eager"

if RUN_TORCH_COMPILE and device.type == "cuda" and hasattr(torch, "compile"):
    try:
        inference_model = torch.compile(resumed_model, mode="reduce-overhead")
        with torch.inference_mode():
            _ = inference_model(sample_features)  # compile warmup
        compile_status = "compiled"
    except Exception as error:
        inference_model = resumed_model
        compile_status = f"fallback: {type(error).__name__}"
print("compile status:", compile_status)

In [ ]:
@torch.inference_mode()
def evaluate(model: nn.Module, loader: DataLoader) -> dict[str, float]:
    model.eval()
    start = time.perf_counter()
    correct = 0
    examples = 0
    for batch in loader:
        features, labels = move_batch(batch)
        logits = model(features)
        correct += logits.argmax(dim=1).eq(labels).sum().item()
        examples += len(labels)
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - start
    return {"accuracy": correct / examples, "examples_per_second": examples / elapsed}


final_metrics = evaluate(inference_model, validation_loader)
assert 0.0 <= final_metrics["accuracy"] <= 1.0
assert math.isfinite(final_metrics["examples_per_second"])
assert second_epoch_metrics["loss"] <= first_epoch_metrics["loss"] + 0.5
checkpoint_path.unlink(missing_ok=True)
print("validation:", {k: round(v, 3) for k, v in final_metrics.items()})
print("checkpoint cleaned:", not checkpoint_path.exists())

## 마무리 — RTX 3080 실험 체크리스트

- DataLoader worker보다 먼저 `num_workers=0 + pin_memory` 기준선을 측정합니다.
- AMP, batch size, accumulation을 한 번에 바꾸지 말고 throughput/peak VRAM/metric을 함께 기록합니다.
- `reserved > allocated`는 정상일 수 있으며 반복적인 `empty_cache()`를 최적화로 오해하지 않습니다.
- Batch-size 추정표는 1차 필터이고 sequence model은 길이 제곱 activation을 별도 반영합니다.
- Resume 테스트는 새 객체에서 수행하고 RNG/loader/scheduler/tokenizer까지 저장합니다.
- `torch.compile`은 eager 정답 검증 뒤 반복 workload에서만 opt-in benchmark합니다.

이 학습 함수를 06번 classifier, 08번 RNN, 16번 Transformer에 적용하며 모델별 peak 메모리와
examples(or tokens)/sec를 비교하면 노트북 사양에 맞는 실전 기준선을 만들 수 있습니다.